In [1]:
# -----------------------------
# PART 0 — Setup + locate files
# -----------------------------
from pathlib import Path

BASE = Path(r"C:\Users\timil\Downloads\Joke_Project_Diss2026")

print("\n==============================")
print("PART 0 — Setup + locate files")
print("==============================")
print("Base folder:", BASE)
print("Exists?:", BASE.exists())

print("\nCSV files in folder:")
for p in sorted(BASE.glob("*.csv")):
    print(" -", p.name)



PART 0 — Setup + locate files
Base folder: C:\Users\timil\Downloads\Joke_Project_Diss2026
Exists?: True

CSV files in folder:
 - ShortJokes-Userdata-edges-A.csv
 - ShortJokes-Userdata-edges-B.csv
 - ShortJokes-Userdata-edges-C.csv
 - ShortJokes-Userdata-edges-D.csv
 - shortjokes.csv
 - shortjokes_edges_clean.csv
 - shortjokes_edges_with_ids.csv
 - shortjokes_item_map.csv
 - shortjokes_ratings_combined.csv
 - shortjokes_user_map.csv


In [2]:
# -----------------------------
# PART 1 — Load full Short Jokes + ABCD ratings
# -----------------------------
import pandas as pd

print("\n========================================")
print("PART 1 — Load full Short Jokes + ratings")
print("========================================")

full_path = BASE / "shortjokes.csv"
ratings_path = BASE / "shortjokes_ratings_combined.csv"

df_full = pd.read_csv(full_path)
df_ratings = pd.read_csv(ratings_path)

print("Loaded full jokes:", full_path.name, "| rows:", len(df_full), "| cols:", list(df_full.columns))
print("Loaded ratings:   ", ratings_path.name, "| rows:", len(df_ratings), "| cols:", list(df_ratings.columns))

print("\nFull jokes preview:")
display(df_full.head())

print("\nRatings preview:")
display(df_ratings.head())



PART 1 — Load full Short Jokes + ratings
Loaded full jokes: shortjokes.csv | rows: 231657 | cols: ['ID', 'Joke']
Loaded ratings:    shortjokes_ratings_combined.csv | rows: 196 | cols: ['user_id', 'joke_id', 'joke_text', 'rating', 'source_file']

Full jokes preview:


,ID,Joke
0,1,"[me narrating a documentary about narrators] ""..."
1,2,Telling my daughter garlic is good for you. Go...
2,3,I've been going through a really rough period ...
3,4,"If I could have dinner with anyone, dead or al..."
4,5,Two guys walk into a bar. The third guy ducks.



Ratings preview:


,user_id,joke_id,joke_text,rating,source_file
0,A,15270,Where does beef come from? Cowschwitz.,1,ShortJokes-Userdata-edges-A.csv
1,A,28989,What do Angels fans and gay men both have in c...,0,ShortJokes-Userdata-edges-A.csv
2,A,40594,What's the difference between a garbanzo bean ...,-1,ShortJokes-Userdata-edges-A.csv
3,A,41814,Trying to take the best instagram picture ever...,1,ShortJokes-Userdata-edges-A.csv
4,A,46109,Chess makes us to realize our life!!! Chess sa...,1,ShortJokes-Userdata-edges-A.csv


In [3]:
# -----------------------------
# PART 2 — Clean full jokes + align IDs with ratings
# -----------------------------
print("\n========================================")
print("PART 2 — Clean full jokes + align IDs")
print("========================================")

# Make sure IDs are strings in BOTH tables (so merges/lookup work)
df_full["ID"] = df_full["ID"].astype(str).str.strip()
df_ratings["joke_id"] = df_ratings["joke_id"].astype(str).str.strip()
df_ratings["user_id"] = df_ratings["user_id"].astype(str).str.strip()

# Clean joke text (full dataset)
df_full["Joke"] = df_full["Joke"].astype(str).str.strip()

# Drop empty jokes + duplicates (keeps TF-IDF cleaner and faster)
before = len(df_full)
df_full = df_full[df_full["Joke"].str.len() > 0].copy()
df_full = df_full.drop_duplicates(subset=["Joke"]).reset_index(drop=True)
after = len(df_full)

print(f"Full jokes: {before} -> {after} after removing empty/duplicates")

# Quick check: how many of your rated joke_ids exist in the full dataset?
rated_ids = set(df_ratings["joke_id"].tolist())
full_ids = set(df_full["ID"].tolist())
matched = len(rated_ids.intersection(full_ids))

print("Rated joke_ids:", len(rated_ids))
print("Matched rated IDs found in full dataset:", matched)



PART 2 — Clean full jokes + align IDs
Full jokes: 231657 -> 231613 after removing empty/duplicates
Rated joke_ids: 49
Matched rated IDs found in full dataset: 49


In [4]:
# -----------------------------
# PART 3 — Build TF-IDF vectors (ALL jokes)
# -----------------------------
print("\n========================================")
print("PART 3 — Build TF-IDF vectors (ALL jokes)")
print("========================================")

from sklearn.feature_extraction.text import TfidfVectorizer

# Explanation:
# - TF-IDF converts each joke into a numeric vector of weighted words/phrases.
# - stop_words removes common words (helps short text)
# - ngram_range=(1,2) captures single words + short phrases (often useful for jokes)
# - min_df=2 removes very rare terms (reduces noise)
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = vectorizer.fit_transform(df_full["Joke"])

print("TF-IDF matrix built.")
print("Shape (n_jokes x n_features):", tfidf_matrix.shape)



PART 3 — Build TF-IDF vectors (ALL jokes)
TF-IDF matrix built.
Shape (n_jokes x n_features): (231613, 219679)


In [5]:
# -----------------------------
# PART 4 — Build user profiles (likes-only, from ABCD)
# -----------------------------
print("\n========================================")
print("PART 4 — Build user profiles (likes-only)")
print("========================================")

import numpy as np

# Explanation:
# - We take only the jokes the user liked (rating==1).
# - A user's profile is the AVERAGE TF-IDF vector of those liked jokes.
df_ratings["user_id"] = df_ratings["user_id"].astype(str).str.strip()
df_ratings["joke_id"] = df_ratings["joke_id"].astype(str).str.strip()

likes_df = df_ratings[df_ratings["rating"] == 1].copy()
print("Total likes (rating==1):", len(likes_df))

# Map full dataset ID -> row index so we can grab TF-IDF rows quickly
id_to_row = {jid: i for i, jid in enumerate(df_full["ID"].tolist())}

users = sorted(likes_df["user_id"].unique())
print("Users found:", users)

def build_like_profile(user_id):
    liked_ids = likes_df.loc[likes_df["user_id"] == user_id, "joke_id"].tolist()
    liked_rows = [id_to_row[jid] for jid in liked_ids if jid in id_to_row]

    if len(liked_rows) == 0:
        return None, 0

    # mean() can return a numpy.matrix, which breaks sklearn's linear_kernel
    profile_vec = tfidf_matrix[liked_rows].mean(axis=0)
    profile_vec = np.asarray(profile_vec)  # convert to normal (1, n_features) array

    return profile_vec, len(liked_rows)


user_profiles = {}
for u in users:
    profile, count = build_like_profile(u)
    user_profiles[u] = profile
    print(f"User {u}: liked jokes matched in full dataset = {count}")



PART 4 — Build user profiles (likes-only)
Total likes (rating==1): 86
Users found: ['A', 'B', 'C', 'D']
User A: liked jokes matched in full dataset = 28
User B: liked jokes matched in full dataset = 16
User C: liked jokes matched in full dataset = 25
User D: liked jokes matched in full dataset = 17


In [6]:
# -----------------------------
# PART 5 — Recommend Top-K (TF-IDF cosine similarity)
# -----------------------------
print("\n========================================")
print("PART 5 — Recommend Top-K (TF-IDF similarity)")
print("========================================")

from sklearn.metrics.pairwise import linear_kernel

# Explanation:
# - cosine similarity tells us how close a candidate joke is to the user's profile.
# - We also filter out jokes the user already rated (so they get new ones).
def recommend_topk(user_id, k=5):
    profile = user_profiles.get(user_id, None)
    if profile is None:
        return pd.DataFrame()

    sims = linear_kernel(profile, tfidf_matrix).ravel()

    # Filter out jokes this user already rated (any rating)
    seen_ids = set(df_ratings.loc[df_ratings["user_id"] == user_id, "joke_id"].tolist())
    seen_rows = [id_to_row[jid] for jid in seen_ids if jid in id_to_row]
    sims[seen_rows] = -1.0

    top_idx = np.argsort(-sims)[:k]
    recs = df_full.iloc[top_idx][["ID", "Joke"]].copy()
    recs["score"] = sims[top_idx]
    return recs.reset_index(drop=True)

# Demo output
K = 5
for u in users:
    print("\n----------------------------------------")
    print(f"User {u} — Top-{K} TF-IDF recommendations")
    print("----------------------------------------")

    recs = recommend_topk(u, k=K)
    for i, row in recs.iterrows():
        print(f"{i+1}. (ID={row['ID']}, score={row['score']:.3f}) {row['Joke'][:140]}{'...' if len(row['Joke'])>140 else ''}")



PART 5 — Recommend Top-K (TF-IDF similarity)

----------------------------------------
User A — Top-5 TF-IDF recommendations
----------------------------------------
1. (ID=109548, score=0.038) Donald Trump Running For President
2. (ID=92098, score=0.036) Q: What do you call a nun in a wheelchair? A: Virgin Mobile
3. (ID=32303, score=0.036) What do you call a nun in a wheelchair? Virgin mobile
4. (ID=146499, score=0.036) What do you call a nun in a wheelchair Virgin Mobile
5. (ID=80373, score=0.036) What do you call a nun in a wheelchair... Virgin mobile

----------------------------------------
User B — Top-5 TF-IDF recommendations
----------------------------------------
1. (ID=109548, score=0.067) Donald Trump Running For President
2. (ID=129092, score=0.062) What do you call a Nun in a Wheelchair? Virgin Mobile
3. (ID=80373, score=0.062) What do you call a nun in a wheelchair... Virgin mobile
4. (ID=88825, score=0.062) What do you call a nun in a wheelchair Virgin mobile
5. (ID=15

In [7]:
# -----------------------------
# PART 5B — Diversity filter (reduce near-duplicate recommendations)
# -----------------------------
print("\n====================================================")
print("PART 5B — Diversity filter (reduce near-duplicates)")
print("====================================================")

import numpy as np
from sklearn.metrics.pairwise import linear_kernel

# Explanation (simple):
# 1) Get a larger candidate pool (e.g., top 200 by similarity).
# 2) Build the final Top-K by skipping jokes that are too similar to ones already chosen.
# This stops the output being "the same joke 5 times" with tiny wording differences.

def recommend_topk_diverse(user_id, k=5, candidate_pool=200, max_sim=0.75):
    """
    TF-IDF recommender with a diversity filter.
    - candidate_pool: how many top-scoring jokes to consider before filtering
    - max_sim: if a candidate is too similar to an already-picked joke, skip it
    """
    profile = user_profiles.get(user_id, None)
    if profile is None:
        return pd.DataFrame()

    # Similarity user profile -> all jokes
    sims = linear_kernel(profile, tfidf_matrix).ravel()

    # Filter out jokes the user already rated (any rating)
    seen_ids = set(df_ratings.loc[df_ratings["user_id"] == user_id, "joke_id"].tolist())
    seen_rows = [id_to_row[jid] for jid in seen_ids if jid in id_to_row]
    sims[seen_rows] = -1.0

    # Take a larger candidate pool first
    candidate_idx = np.argsort(-sims)[:candidate_pool]

    picked = []
    for idx in candidate_idx:
        if len(picked) >= k:
            break

        # First pick always accepted
        if len(picked) == 0:
            picked.append(idx)
            continue

        # Check similarity to already picked jokes (item-to-item similarity)
        # We compare TF-IDF vectors directly:
        # if candidate is too similar to ANY picked joke, skip it.
        cand_vec = tfidf_matrix[idx]
        picked_vecs = tfidf_matrix[picked]
        max_pair_sim = linear_kernel(cand_vec, picked_vecs).max()

        if max_pair_sim <= max_sim:
            picked.append(idx)

    # If we filtered too aggressively and didn't get k items, fill the rest without filtering
    if len(picked) < k:
        for idx in candidate_idx:
            if len(picked) >= k:
                break
            if idx not in picked:
                picked.append(idx)

    recs = df_full.iloc[picked][["ID", "Joke"]].copy()
    recs["score"] = sims[picked]
    return recs.reset_index(drop=True)


# ---- Demo output (diverse Top-5) ----
K = 5
CAND_POOL = 200
MAX_SIM = 0.75  # try 0.70 (more diverse) or 0.85 (less strict) if needed

print(f"Settings: K={K}, candidate_pool={CAND_POOL}, max_sim={MAX_SIM}")

for u in users:
    print("\n----------------------------------------")
    print(f"User {u} — Diverse Top-{K} TF-IDF recommendations")
    print("----------------------------------------")

    recs = recommend_topk_diverse(u, k=K, candidate_pool=CAND_POOL, max_sim=MAX_SIM)
    for i, row in recs.iterrows():
        print(f"{i+1}. (ID={row['ID']}, score={row['score']:.3f}) {row['Joke'][:140]}{'...' if len(row['Joke'])>140 else ''}")

print("\n✅ PART 5B complete: recommendations should contain fewer near-duplicates.")



PART 5B — Diversity filter (reduce near-duplicates)
Settings: K=5, candidate_pool=200, max_sim=0.75

----------------------------------------
User A — Diverse Top-5 TF-IDF recommendations
----------------------------------------
1. (ID=109548, score=0.038) Donald Trump Running For President
2. (ID=92098, score=0.036) Q: What do you call a nun in a wheelchair? A: Virgin Mobile
3. (ID=193157, score=0.034) God promised men that good and obedient wives would be found in all corners of the world. Then he made the earth round.......and laughed and...
4. (ID=146140, score=0.030) How do you get Lady Gaga to leave you alone? You Poke-r Face.
5. (ID=68322, score=0.029) God promised men that good obedient wives would be found in all 4 corners of the world, then he made the world round. What a funny guy

----------------------------------------
User B — Diverse Top-5 TF-IDF recommendations
----------------------------------------
1. (ID=109548, score=0.067) Donald Trump Running For President
2. (